In [2]:
import json
spkrs, txts, ner = [], [], []
with open("yes_self_ner.txt") as file:
    while True:
        speech_ln = file.readline()
        ner_ln = file.readline()
        while ner_ln and ner_ln[0] != "[":
            speech_ln += ner_ln
            ner_ln = file.readline()
        if not ner_ln:
            break
        ner_ln = ner_ln.replace("\'", "\"")
        splits = speech_ln.split(":", 1)
        if len(splits) != 2:
            continue
        speaker, text = splits
        if "Committee" not in speaker and len(speaker) > 1:
            spkrs.append(speaker.strip())
            txts.append(text.strip())
            ner.append(json.loads(ner_ln))

In [3]:
print(len(spkrs))

9921


In [5]:
data = list(zip(spkrs, txts, ner))
data[2]

('Tam Ma',
 "Hi, good morning, Mr. Chairs and Members. Tam Ma with Health Access California, the State Healthcare Consumer Advocacy Coalition. We worked very closely with Representative Becerra and his office to ensure passage of the Affordable Care Act and we have the utmost confidence that he will continue working hard to ensure that all Californians have access to quality and affordable health care and on a personal note, I grew up in Mr. Becerra's district, and my mother still resides there and I'm very honored to be here today in support of his nomination. Thank you.",
 [{'start': 22, 'end': 28, 'label': 'PERSON'},
  {'start': 42, 'end': 48, 'label': 'SPEAKER'},
  {'start': 173, 'end': 180, 'label': 'PERSON'},
  {'start': 433, 'end': 440, 'label': 'PERSON'}])

In [11]:
training_data = {'classes': ['PERSON', 'SPEAKER'], 'annotations': []}
for i in data:
    temp_dict = {}
    temp_dict['text'] = i[1]
    temp_dict['entities'] = []
    for tag in i[2]:
      temp_dict['entities'].append((tag['start'], tag['end'], tag['label']))
    training_data['annotations'].append(temp_dict)

training_data['annotations'][0]

{'text': "Thank you, Speaker, and the members of the Assembly. I'm Steven Choi, as you already heard the last time. As a Korean-descent member of the community, when I hear that Assemblyman Matthew Harper proposed to adjourn our session in memory of Dr. Sammy Lee, who was a Korean American, and myself, I first of all appreciate for that suggestion and I must add my few comments, as I know he was such a great role model, a hero to our community. I know he was well-known as a Olympic gold medalist, as a physician, and he served his community for a long time. I met him several times when he came to the City of Irvine in commemoration of Korean American Day. He's full of humor, wit, and knowledge and every time he speaks, people crack up with laughs. So such a wonderful human being. And then also, we recognize him as a very small physical stature, however, he was a giant to our Korean American community. He continues to serve as a role model of our community. So this is very proper, and I 

In [25]:
import spacy
from spacy.tokens import DocBin
from tqdm import tqdm

nlp = spacy.blank("en") # load a new spacy model

In [26]:
from spacy.util import filter_spans

def save_spacy_data(data, output):
    doc_bin = DocBin() # create a DocBin object
    entities_skipped = 0
    for training_example in data: 
        text = training_example['text']
        labels = training_example['entities']
        doc = nlp.make_doc(text) 
        ents = []
        for start, end, label in labels:
            span = doc.char_span(start, end, label=label, alignment_mode="contract")
            if span is None:
                print("Skipping entity")
                entities_skipped += 1
                continue
            else:
                ents.append(span)
        filtered_ents = filter_spans(ents)
        doc.ents = filtered_ents 
        doc_bin.add(doc)

    print(f"Total entities skipped: {entities_skipped}")
    doc_bin.to_disk(output) # save the docbin object

In [ ]:
split_ratio = 0.8
split_index = int(len(training_data) * split_ratio)
TRAINING_DATA = training_data['annotations'][:split_index]
DEV_DATA = training_data['annotations'][split_index:]

save_spacy_data(TRAINING_DATA, "train.spacy")
save_spacy_data(DEV_DATA, "dev.spacy")

Total entities skipped: 0
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entit

In [10]:
spacy.load('en_core_web_lg')

In [ ]:
nlp_ner = spacy.load("model-best")

In [13]:
doc = nlp_ner("Hi, good morning, Mr. Chairs and Members. Tam Ma with Health Access California, the State Healthcare Consumer Advocacy Coalition. We worked very closely with Representative Becerra and his office to ensure passage of the Affordable Care Act and we have the utmost confidence that he will continue working hard to ensure that all Californians have access to quality and affordable health care and on a personal note, I grew up in Mr. Becerra's district, and my mother still resides there and I'm very honored to be here today in support of his nomination. Thank you.")

colors = {"PERSON": "#F67DE3", "SPEAKER": "#7DF6D9"}
options = {"colors": colors} 

spacy.displacy.render(doc, style="ent", options= options, jupyter=True)